In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t

sys.path.append("..")
from modeling import bh_fdr

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
OUT_DIR = DATA_DIR / "train_relationship"
PATTERNS = [
    "vegan_score_per_day", "vegetarian_score_per_day", "pescatarian_score_per_day", "wfpb_score_per_day",
    "carnivore_score_per_day", "NOVA_pct", "low_carb_diet", "extremely_low_carb", "eatLANCET_score_per_day",
    "IMEDAS_score_per_day", "DASH_score", "aHEI_score_per_day", "hPDI_score_per_day",
]
PRIMARY_COVARIATES = ["age", "sex", "bmi", "smoking_binary"]
EXTENDED_COVARIATES = PRIMARY_COVARIATES + ["oralhealth_soft_tissue_inflammation_level", "oralhealth_periodontal_damage_level"]


def read(path):
    df = pd.read_csv(path, low_memory=False)
    df["participant_id"] = df["participant_id"].astype(str)
    return df


def ols_effect(data, exposure, outcome, covariates):
    sub = data[[exposure] + covariates + [outcome]].dropna()
    if len(sub) <= len(covariates) + 5:
        return len(sub), np.nan, np.nan
    x = np.column_stack([np.ones(len(sub)), sub[[exposure] + covariates].to_numpy(dtype=float)])
    y = sub[outcome].to_numpy(dtype=float)
    beta = np.linalg.lstsq(x, y, rcond=None)[0]
    residual = y - x @ beta
    df_resid = len(y) - x.shape[1]
    se = np.sqrt(max((residual @ residual) / df_resid * np.linalg.pinv(x.T @ x)[1, 1], 0))
    return len(sub), float(beta[1]), float(2 * t.sf(abs(beta[1] / se), df_resid)) if se > 0 else np.nan


def associations(data, outcomes, covariates):
    rows = [(pattern, outcome, *ols_effect(data, pattern, outcome, covariates))
            for pattern in patterns for outcome in outcomes]
    result = pd.DataFrame(rows, columns=["diet_pattern", "outcome", "n", "beta", "p_value"])
    result["fdr_q"] = bh_fdr(result["p_value"])
    return result


daily_scores = read(DATA_DIR / "diet_experiment" / "diet_daily_scores_baseline.csv")
patterns = [f"{c}_mean" for c in PATTERNS]
pattern_means = daily_scores.groupby("participant_id")[PATTERNS].mean().add_suffix("_mean").reset_index()
covariates = read(DATA_DIR / "covariates" / "covariates_age_sex_bmi_smoking_oralhealth_baseline.csv")
summary = pd.read_csv(DATA_DIR / "train" / "model_species_abundance_summary_baseline.csv")
species_log10 = read(DATA_DIR / "oral_experiment" / "oral_species_relative_log10_baseline.csv")
diversity = read(DATA_DIR / "oral_experiment" / "oral_diversity_baseline.csv")
species = [s for s in summary.loc[summary["kept_for_modeling"], "species"] if s in species_log10]

In [ ]:
species_data = pattern_means.merge(species_log10[["participant_id"] + species], on="participant_id").merge(
    covariates[["participant_id"] + EXTENDED_COVARIATES], on="participant_id", how="left")
diversity_data = pattern_means.merge(diversity[["participant_id", "Richness", "Shannon_diversity"]], on="participant_id").merge(
    covariates[["participant_id"] + EXTENDED_COVARIATES], on="participant_id", how="left")

OUT_DIR.mkdir(parents=True, exist_ok=True)
for outcome_set, data, outcomes in [("species", species_data, species),
                                     ("diversity", diversity_data, ["Richness", "Shannon_diversity"])]:
    for model, cov in [("primary", PRIMARY_COVARIATES), ("extended", EXTENDED_COVARIATES)]:
        result = associations(data, outcomes, cov)
        result.to_csv(OUT_DIR / f"diet_pattern_{outcome_set}_{model}_model.csv", index=False)
        print(outcome_set, model, "FDR-significant associations:", int((result["fdr_q"] < 0.05).sum()))